**Mini Project**
Airline Tweet Sentiment Classifier using Natural Language Processing**
---
**Notes:**
Use sample dataset:
https://github.com/salman1256/aimltraining/blob/main/Day-30/airline_tweets_sample.csv

---
# Steps:
1. Import libraries  
2. Load and explore dataset  
3. Clean and preprocess the text  
4. Convert text to numerical vectors (TF-IDF)  
5. Split into train and test sets  
6. Train a Logistic Regression model  
7. Evaluate accuracy and classification report  
8. Predict sentiment for new example tweets  






In [227]:
# Step 1 a: Import Required Libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
import re
import nltk
from nltk.corpus import stopwords
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer


In [228]:
# Step 1 b: Download nltk required thing like stopwords
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Faridah\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Faridah\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Faridah\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Faridah\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Faridah\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [229]:
#Step 2 a: Load DataSet
df = pd.read_csv("airline_tweets_sample.csv")

In [230]:
#Step 2 : Check Data set  at least top 5 values
df.head()

,text,sentiment
0,"@United flight was delayed for 3 hours, worst ...",negative
1,"Loved the service on @Delta, crew was super fr...",positive
2,"@AmericanAir lost my luggage again, so disappo...",negative
3,Smooth boarding and on-time arrival. Great job...,positive
4,The seats were uncomfortable but staff was polite,neutral


In [231]:
# Step 3:
#Text Cleaning and Preprocessing
# For each tweet do:
# a. Convert to lowercase
# b. Remove URLs
# c. Remove special characters and numbers
# d. Remove stopwords (common words like *the, is, and* etc.)
# e. Apply **stemming** (reduce words to their root form)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [word for word in words if word not in stopwords.words('english')]
    words = [stemmer.stem(word) for word in words]
    text = ' '.join(words)
    return text

df['clean_tweet']=df['text'].apply(clean_text)
df

,text,sentiment,clean_tweet
0,"@United flight was delayed for 3 hours, worst ...",negative,unit flight delay hour worst experi ever
1,"Loved the service on @Delta, crew was super fr...",positive,love servic delta crew super friendli
2,"@AmericanAir lost my luggage again, so disappo...",negative,americanair lost luggag disappoint
3,Smooth boarding and on-time arrival. Great job...,positive,smooth board ontim arriv great job southwestair
4,The seats were uncomfortable but staff was polite,neutral,seat uncomfort staff polit
5,"@JetBlue flight attendants were rude, not flyi...",negative,jetblu flight attend rude fli
6,"Got a free upgrade to business class, thank yo...",positive,got free upgrad busi class thank unit
7,"Average flight, nothing special to mention",neutral,averag flight noth special mention
8,@DeltaAirLines provided excellent support with...,positive,deltaairlin provid excel support book
9,The in-flight entertainment was not working,negative,inflight entertain work


In [232]:
#Step: 4
#a) Convert text to numerical vectors (TF-IDF)
# b) check x,y and shape len
vectorizer = TfidfVectorizer()
x = vectorizer.fit_transform(df['clean_tweet'])
y = df['sentiment']
print(x)
print(y)
print("Shape of x:", x.shape)      
print("Length of y:", len(y)) 

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 140 stored elements and shape (30, 105)>
  Coords	Values
  (0, 98)	0.3804945349431948
  (0, 41)	0.2838360742053801
  (0, 26)	0.34767619843477227
  (0, 51)	0.4267493823409172
  (0, 103)	0.3804945349431948
  (0, 37)	0.3804945349431948
  (0, 34)	0.4267493823409172
  (1, 60)	0.3917559302157924
  (1, 85)	0.3917559302157924
  (1, 27)	0.4393797699957502
  (1, 24)	0.3917559302157924
  (1, 91)	0.3917559302157924
  (1, 44)	0.4393797699957502
  (2, 3)	0.4576898872817969
  (2, 59)	0.513328993622341
  (2, 61)	0.513328993622341
  (2, 30)	0.513328993622341
  (3, 86)	0.36253162556198376
  (3, 10)	0.33126262222553676
  (3, 72)	0.4066028104995579
  (3, 5)	0.4066028104995579
  (3, 48)	0.36253162556198376
  (3, 54)	0.4066028104995579
  (3, 88)	0.36253162556198376
  (4, 84)	0.4610552498655506
  :	:
  (24, 55)	0.41617114486261303
  (24, 99)	0.46676302208846976
  (24, 4)	0.46676302208846976
  (24, 23)	0.46676302208846976
  (25, 43)	0.4257896469664

In [233]:
# Step 5: Split into train and test sets
# 80% training and 20% testing
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [234]:
# Step:6 Train a Logistic Regression model
# a: Create Logistic Model
# b: Train Logistic Model
model=LogisticRegression(max_iter=200, class_weight='balanced')
model.fit(x_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,200
,multi_class,'deprecated'


In [235]:
#Step 7:
# Evaluate accuracy and classification report
# a. predict Model
y_pred = model.predict(x_test)
     
# b. Precision, reacll, F1-Score for each sentiments
print(classification_report(y_test, y_pred, target_names=['negative', 'neutral', 'positive']))


              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         1
     neutral       0.00      0.00      0.00         1
    positive       0.50      0.50      0.50         4

    accuracy                           0.33         6
   macro avg       0.17      0.17      0.17         6
weighted avg       0.33      0.33      0.33         6



C:\Users\Faridah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Faridah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Faridah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [236]:
#Step 8: Predict sentiment for new example tweets

new_tweets = [
    "Flight delays and cancellations.",
    "Poor website usability or digital service problems.",
    "Great service and smooth boarding!"
]

clean_tweets = [clean_text(t) for t in new_tweets]
tfidf_new = vectorizer.transform(clean_tweets)
preds = model.predict(tfidf_new)

for twt, pred in zip(new_tweets, preds):
    print(f"{twt} ---> {pred}")

Flight delays and cancellations. ---> neutral
Poor website usability or digital service problems. ---> negative
Great service and smooth boarding! ---> positive
